# 00. v0 → v1 migration guide

Covers the breaking changes and code mapping you need to know when transitioning from LangChain/LangGraph v0 to v1.

## Learning Objectives

- Understand v1 package structure changes and import paths
- Perform `create_react_agent` → `create_agent` migration
- Apply middleware-based dynamic prompting, state management, and context injection
- Utilizes standard content blocks and structured output strategies

## 0.1 Change package structure

In v1, the `langchain` namespace has been significantly reduced to five core modules essential for building an agent:

| v1 module | Role | Main API |
|---|---|---|
| `langchain.agents` | Agent creation and state management | `create_agent`, `AgentState` |
| `langchain.messages` | Message types and content blocks | `HumanMessage`, `AIMessage`, `content_blocks` |
| `langchain.tools` | tool Definition | `@tool` decorator, `BaseTool` |
| `langchain.chat_models` | model initialization | `init_chat_model` |
| `langchain.embeddings` | Embedding Utility | Embedding model wrapper |

### Legacy code migration — `langchain-classic`

Chains, Retrievers, Hub, and Indexing API, which were previously used in the `langchain` package, have all been separated into a separate package called `langchain-classic`. If you need to keep your existing code, change the import path after installation to `pip install langchain-classic`:

The ```python
# v0 — 기존 방식
from langchain.chains import LLMChain
from langchain.retrievers import MultiQueryRetriever
from langchain import hub

# v1 — langchain-classic으로 이전
from langchain_classic.chains import LLMChain
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_classic import hub
```

이 분리를 통해 v1의 `langchain` package has become a lightweight structure that focuses only on agent building, and legacy functionality is maintained independently.

## 0.2 Agent creation API changes

In [ ]:
from dotenv import load_dotenv
load_dotenv(override=True)

from langchain_openai import ChatOpenAI
from langchain.tools import tool
from langchain.agents import create_agent

model = ChatOpenAI(model="gpt-4.1")

@tool
def add(a: int, b: int) -> int:
    """Add two numbers together."""
    return a + b

# v0: from langgraph.prebuilt import create_react_agent
# v1: from langchain.agents import create_agent
agent = create_agent(
    model=model,
    tools=[add],
    system_prompt="You are a math assistant.",  # v0: prompt=
)
print("✓ v1 agent creation completed")

In [ ]:
# Observability settings (optional) - LangSmith or Langfuse
# Set the key in .env, or uncomment it below and enter it yourself.
# os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-..."
# os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-..."
# os.environ["LANGFUSE_HOST"] = "https://lf.ddok.ai"
import os

# LangSmith: Automatically enabled when LANGSMITH_TRACING=true (no code modification required)
if os.environ.get("LANGSMITH_TRACING", "").lower() == "true":
    os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")
    os.environ.setdefault("LANGCHAIN_API_KEY", os.environ.get("LANGSMITH_API_KEY", ""))
    os.environ.setdefault("LANGCHAIN_PROJECT", os.environ.get("LANGSMITH_PROJECT", "default"))
    print(f"LangSmith tracing ON \u2014 project: {os.environ['LANGCHAIN_PROJECT']}")

# Langfuse: Pass config={"callbacks": [langfuse_handler]} when calling invoke/stream
langfuse_handler = None
if os.environ.get("LANGFUSE_SECRET_KEY"):
    from langfuse.langchain import CallbackHandler
    langfuse_handler = CallbackHandler()
    print(f"Langfuse tracing ON \u2014 {os.environ.get('LANGFUSE_HOST', '')}")

# Langfuse config: pass to invoke/stream/batch calls
lf_config = {"callbacks": [langfuse_handler]} if langfuse_handler else {}


### Major changes

| v0 | v1 | Remarks |
|---|---|---|
| `from langgraph.prebuilt import create_react_agent` | `from langchain.agents import create_agent` | import path + function name |
| `prompt=` | `system_prompt=` | Parameter name |
| `ToolNode` Support | Not supported | Functions/BaseTool/dict only |
| `pre_hooks`, `post_hooks` | `middleware=[]` | Integrated middleware |
| Pydantic/dataclass status | `TypedDict` only | state schema |

## 0.3 State Schema — TypedDict only

In v1, custom state must inherit from `TypedDict` based on `AgentState`.

In [ ]:
from langchain.agents import AgentState, create_agent

class CustomState(AgentState):
    user_id: str

agent = create_agent(
    model=model,
    tools=[add],
    state_schema=CustomState,
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "How much is 3+4?"}],
    "user_id": "user_123",
}, config=lf_config)
print(result["messages"][-1].content)

## 0.4 Runtime context injection (new)

In v1, **immutable runtime data** can be passed to the agent via the `context_schema` and `context` parameters. This is a pattern that safely passes data that varies from request to request, such as user ID, role, and session information, but does not change during execution, to the agent and tool.

**How it works:**
1. Define the context schema with `@dataclass`.
2. Register the schema with `create_agent(context_schema=...)`.
3. Pass the value to runtime with `agent.invoke(..., context=ContextInstance(...))`.
4. In tool, the context is accessed with the `ToolRuntime[ContextType]` parameter.

Context is read-only data that **does not change** between tool calling, unlike agent state (`AgentState`). The state is updated during the agent loop, but the context is fixed when calling `invoke`.

In [ ]:
from dataclasses import dataclass
from langchain.tools import tool, ToolRuntime

@dataclass
class UserContext:
    user_id: str
    role: str

@tool
def get_role(runtime: ToolRuntime[UserContext]) -> str:
    """Query the current user's roles."""
    return f"사용자 {runtime.context.user_id}의 역할: {runtime.context.role}"

agent = create_agent(
    model=model,
    tools=[get_role],
    context_schema=UserContext,
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "What is my role?"}]},
    context=UserContext(user_id="u1", role="admin"),
    config=lf_config,
)
print(result["messages"][-1].content)

## 0.5 Dynamic Prompts — Middleware Approach (New)

Instead of the static prompt in v0, `@dynamic_prompt` creates a prompt dynamically based on the runtime context.

In [ ]:
from langchain.agents.middleware import dynamic_prompt, ModelRequest

@dynamic_prompt
def role_based_prompt(request: ModelRequest) -> str:
    role = request.runtime.context.role
    if role == "admin":
        return "You are an administrator assistant with full access."
    return "You are a read-only assistant."

agent = create_agent(
    model=model,
    tools=[add],
    context_schema=UserContext,
    middleware=[role_based_prompt],
)

result = agent.invoke(
    {"messages": [{"role": "user", "content": "hello!"}]},
    context=UserContext(user_id="u1", role="admin"),
    config=lf_config,
)
print(result["messages"][-1].content)

## 0.6 tool Error handling — `@wrap_tool_call` (new)

Instead of v0's `handle_tool_errors`, v1 handles tool errors with middleware.

In [ ]:
from langchain.agents.middleware import wrap_tool_call
from langchain.messages import ToolMessage

@wrap_tool_call
def handle_errors(request, handler):
    try:
        return handler(request)
    except Exception as e:
        return ToolMessage(
            content=f"도구 오류: {e}",
            tool_call_id=request.tool_call["id"],
        )

agent = create_agent(
    model=model,
    tools=[add],
    middleware=[handle_errors],
)
print("✓ Application of error handling middleware")

## 0.7 Standard Content Block & structured output (New)

In v1, messages support provider-agnostic `content_blocks`.
structured output was split into two branches: `ToolStrategy` (based on tool calling) and `ProviderStrategy` (native).

In [ ]:
# Standard content block

response = model.invoke(
    "Please say hello.",
    config=lf_config
)

for block in response.content_blocks:
    print(f"  [{block['type']}] {block.get('text', '')[:100]}")

# .text property (v0: .text() method → ​​v1: .text property)

print(f"\n.text: {response.text[:100]}")

## 0.8 Streaming changes

| Item | v0 | v1 |
|---|---|---|
| Agent node name | `"agent"` | `"model"` |
| `.text` | method `.text()` | Property `.text` |

In [ ]:
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "How much is 5+3?"}]},
    context=UserContext(user_id="u1", role="admin"),
    stream_mode="updates",
    config=lf_config,
):
    for node_name, update in chunk.items():
        # v1: node_name is "model" (in v0 it is "agent")
        if "messages" in update:
            last = update["messages"][-1]
            if hasattr(last, "content") and last.content:
                print(f"[{node_name}] {last.content[:200]}")

## 0.9 Guitar Breaking Change

| change | Description |
|---|---|
| **Python 3.10+** | All LangChain packages require Python 3.10 or higher. Versions 3.9 and below are not supported. |
| **return type** | The return type of the chat model has been fixed from `BaseMessage` to `AIMessage`. |
| **OpenAI Responses API** | Message content is in standard block format by default. You can restore the previous behavior with `output_version="v0"`. |
| **Anthropic max_tokens** | Default value changed from 1024 to automatic setting per model. |
| **AIMessage.example** | The `example` parameter has been removed. Use `additional_kwargs`. |
| **AIMessageChunk** | Added `chunk_position` attribute (value `'last'` in last chunk). |
| **`.text` Properties** | `.text()` method changed to `.text` property. |
| **File Encoding** | Files open with UTF-8 encoding by default. |

## Summary — Migration Checklist

- [ ] Python 3.10+ confirmed
- [ ] `create_react_agent` → `create_agent` changed
- [ ] `prompt=` → `system_prompt=` changed
- [ ] Convert state schema to `AgentState` based on `TypedDict`
- [ ] `pre_hooks`/`post_hooks` → `middleware=[]`
- [ ] `ToolNode` → Replaced with Function/BaseTool
- [ ] `.text()` → `.text` property
- [ ] Check streaming node name `"agent"` → `"model"`
- [ ] Move legacy imports to `langchain-classic`

### Next Steps
→ **[01_middleware.ipynb](./01_middleware.ipynb)**: v1’s biggest new feature — Deepening the middleware system